# Ch.3  機械学習で品種を分類する

【講師用完全版】

| 項目 | 内容 |
|------|------|
| 使うデータ | ワインデータ 178件・13特徴量・3品種 |
| 演習時間 | 35分（問3まで完了で合格） |
| ゴール | RandomForest でモデルを作り、正解率と特徴量重要度を確認する |

---
### 📌 講師メモ：時間配分と時間が押した場合の対応

| 区分 | 内容 | 目安時間 |
|------|------|---------|
| 座学前半（昼前） | 教師あり学習・X/y・データ分割・過学習 | 15分 |
| 座学後半（昼後） | 決定木・RF・混同行列・評価指標 | 15分 |
| 演習（問1〜問2） | モデル学習・正解率 | 20分 |
| 演習（問3） | 混同行列・特徴量重要度（最重要） | 15分 |
| 問4（発展） | Precision / Recall | 余り次第 |

**時間が押した場合：**
- 問3の特徴量重要度（Q3-2）は Ch.2 の仮説との答え合わせとして必ず実施する
- 問4（Precision/Recall）は Ch.5 発展で扱うため、スキップ可
- **正解率と混同行列まで確認できれば Ch.4 に進んで問題ない**

---
## Copilot の使い方

URL: https://copilot.microsoft.com

Copilot への伝え方のコツ（5点セット）：
```
【知りたいこと】〇〇をしたい / 〇〇を確認したい
【使うライブラリ】scikit-learn / pandas
【データの形】df という DataFrame、178行×14列（数値13列 + 品種名列）
【環境】Python 3.8.6、JupyterLab
【困っていること】どう書けばよいかわからない
```

💡 Ch.3 の一言アドバイス：
「使うクラス名（RandomForestClassifier など）と、X・y の形を具体的に伝えると
 精度の高いコードが返ってきます。」

「モデルを作りたい」ではなく、
「X_train と y_train を使って RandomForestClassifier を学習させたい、どう書くか」のように、
**変数名・クラス名・やりたいこと** をセットで伝えましょう。

---
## 準備  ライブラリとデータを読み込む

⛔ 注意 最初にこのセルを実行してください。

In [ ]:
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import japanize_matplotlib

wine = load_wine()
df   = pd.DataFrame(wine.data, columns=wine.feature_names)
df["品種名"] = ["wine_" + str(t) for t in wine.target]
print(f"行数: {len(df)}  列数: {len(df.columns)}")

---
## なぜ「機械学習」を使うのか

Ch.1・Ch.2 で「alcohol と proline で品種が分かれそう」という仮説を立てました。

人間が「alcohol >= 13.5 かつ proline >= 700 なら wine_0」と手動でルールを作ることも可能ですが：

| 方法 | 問題点 |
|------|--------|
| 人間のルール | 変数が 13 個あると組み合わせが膨大 |
| 人間のルール | 新しいデータに対応できない |
| 機械学習（RF） | 13 変数すべてを使って最適なルールを自動生成 |

今回使う RandomForest：
- 決定木（木の形のルール）をたくさん作って多数決する
- 「どの変数がどれだけ判断に使われたか（特徴量重要度）」がわかる

---
## STEP 1  モデルへの「入力」と「答え」を分ける

機械学習の学習に必要なのは：
- X（特徴量）: モデルが「見る」数値データ → 13 列の数値
- y（ターゲット）: モデルが「当てる」答え → 品種（0/1/2）

---

Q1-1：特徴量（X）とターゲット（y）に分けてみましょう

- X：品種名を除いた数値列 13 個
- y：品種名列

💡 ヒント Copilot への相談例：
「pandas で特定の列を除いた DataFrame と、その列だけを別変数に分けたい」

In [ ]:
# 品種名を除いた数値列を X（特徴量）、品種名列を y（ターゲット）として分ける
X = df.drop("品種名", axis=1)   # 13 列の数値特徴量
y = df["品種名"]                 # 予測する品種ラベル
print(f"X: {X.shape}  y: {y.shape}")

---
Q1-1（追加）  X と y の形を確認しましょう

X と y が正しく分割できているか、行数・列数を数値で確認します。
「X は 178行×13列、y は 178件」になっていれば成功です。

💡 ヒント：「DataFrame と Series の行数・列数を確認する方法」を Copilot に聞いてみましょう。

In [ ]:
# X の行数・列数と y の件数を確認して、正しく分割できているか検証する
print(f"X: {X.shape}  y: {y.shape}")
print(f"X の列名: {list(X.columns)}")

---
### 📊 出力結果の解説

```
X: (178, 13)  y: (178,)
X の列名: ['alcohol', 'malic_acid', ..., 'proline']
```

📌 ポイント：X の列数が 13（品種名を除いた数値列）・y の件数が 178（全データ件数）であれば正しく分割できています。
「品種名列が X に残っていないか」を列名リストで確認するのがポイントです。

✅ （模範解答）X: (178, 13)、y: (178,)。品種名列が X に含まれていないことを確認。

---
### 📊 出力結果の解説

```
X: (178, 13)  y: (178,)
```

| 変数 | 意味 | 形 |
|------|------|---|
| X | モデルが「見る」13 特徴量 | 178行 × 13列 |
| y | モデルが「当てる」品種ラベル | 178行 × 1列 |

📌 ポイント X と y はセットで扱う。行数が同じ（178）であることが必須。

📌 講師メモ: 「X と y の行数が同じかを確認する習慣を付けましょう」と声がけ。

---
Q1-2：データを「学習用」と「テスト用」に分けましょう

モデルは学習に使ったデータでは高い精度が出ます。
未知のデータ（テスト用）でも正しく予測できるかを確認するため、事前に分割します。

学習用：テスト用 = 8：2 に分割してください。

💡 ヒント Copilot への相談例：
「scikit-learn で DataFrame を学習用とテスト用に 80:20 に分割したい」

In [ ]:
# 学習用（80%）とテスト用（20%）にランダムに分割する（再現性のため random_state を固定）
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"学習: {len(X_train)}件  テスト: {len(X_test)}件")

---
Q1-2（追加）  分割後のデータ件数とクラス分布を確認しましょう

80:20 に分割すると、実際に何件が訓練用・何件がテスト用になるかを確認します。
また、訓練データに3品種がバランスよく含まれているかも確認しましょう。
（特定の品種が少ないと、モデルがその品種を学習しにくくなります）

💡 ヒント：「Series の value_counts で各クラスの件数を確認する方法」を Copilot に聞いてみましょう。

In [ ]:
# 分割後の行数を確認し、訓練データの品種ごとの件数バランスをチェックする
print(f"訓練: {X_train.shape[0]}件  テスト: {X_test.shape[0]}件")
print("\n訓練データの品種分布:")
print(pd.Series(y_train).value_counts().sort_index())

---
### 📊 出力結果の解説

```
訓練: 142件  テスト: 36件

訓練データの品種分布:
wine_0    47
wine_1    57
wine_2    38
```

📌 ポイント：3品種がおよそ均等に訓練データに含まれていれば問題ありません。
極端に少ないクラスがあると「そのクラスだけ精度が低い」原因になります（不均衡データ問題）。

✅ （模範解答）wine_0: 47件、wine_1: 57件、wine_2: 38件。バランスは問題なし。

---
### 📊 出力結果の解説

```
学習: 142件  テスト: 36件
```

📌 ポイント random_state=42 を指定すると、毎回同じ分け方になる（再現性の確保）。
テスト用は「モデルが見たことがないデータ」として保管する。
テスト用で高い精度が出れば「汎化性能が高いモデル」と評価できる。

📌 講師メモ: 「なぜデータを分ける必要があるのか？」を問い返し、過学習の概念を簡単に説明。

---
## 問1  モデルを学習させる ★☆☆

実務でのイメージ：
「過去の顧客データ（X_train, y_train）をもとに、新規顧客（X_test）の行動を予測するモデルを作る」

---

Q：RandomForestClassifier でモデルを作り、学習させてください

1. `model = RandomForestClassifier(...)` でモデルを作る
   📌 `(...)` には `n_estimators=100, random_state=42` を入れます。n_estimators はランダムに作る決定木の本数です（多いほど安定）。
   random_state=42 は乱数のシードを固定します（42 に特別な意味はありません）。誰が実行しても同じ結果になるよう再現性を確保するために指定します。
2. `model.fit(X_train, y_train)` で学習させる

💡 ヒント Copilot への相談例：
「scikit-learn の RandomForestClassifier で分類モデルを作って、学習データで学習させたい」

In [ ]:
# RandomForest モデルを作成して、学習データで学習させる
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
print("学習完了")

---
### 📊 出力結果の解説

```
学習完了
```

📌 ポイント `n_estimators=100` は「決定木を 100 本作って多数決する」という設定。
多くすると安定するが計算時間が増える。少なすぎると不安定。100 は標準的な設定。
`random_state=42` で再現性を確保。

📌 講師メモ: 「RandomForest が何をしているか」を 1 分で補足：
「100 本の木それぞれが予測を出して、多数決で最終結果を決める仕組みです」と説明。

✅ （模範解答）`model.fit(X_train, y_train)` でエラーなく「学習完了」と表示されれば成功。

---
問1-b（追加）  モデルの設定を確認しましょう

学習後に「何本の決定木を使ったか」「何個の特徴量を学習したか」を確認します。
📌 `n_features_in_` が **13** になっていれば正解です（品種名列を除いた 13 列で正しく学習できています）。
   もし 14 になっていたら → STEP 1 に戻って X から品種名列を除いてください。

💡 ヒント：「学習済み RandomForest モデルの木の本数・特徴量数を確認する方法」を Copilot に聞いてみましょう。

In [ ]:
# モデルの設定（決定木の本数・学習した特徴量数）を確認する
print(f"決定木の本数（n_estimators）: {model.n_estimators}")
print(f"学習した特徴量の数（n_features_in_）: {model.n_features_in_}")

---
### 📊 出力結果の解説

```
決定木の本数（n_estimators）: 100
学習した特徴量の数（n_features_in_）: 13
```

📌 ポイント：
- `n_estimators=100` → 100本の決定木が作られ、多数決で最終予測を出していることを確認
- `n_features_in_=13` → X の列数（13列）が正しく認識されていることを確認
  14 と表示された場合は、品種名列が X に含まれたまま学習している（要確認）

✅ （模範解答）100 と 13 が表示されれば正しい設定。

📌 講師メモ：「n_estimators を 10 にしたら精度はどう変わる？」と問い返すと RF の仕組みへの理解が深まる。

---
問1-c（追加）  1件だけ取り出して予測してみましょう

fit()（勉強）が終わったら、predict()（試験）を実際に試してみましょう。
テストデータの最初の1件を取り出して、モデルが何品種と予測するかを確認します。

📌 スライドの「試験勉強（fit）→ 本番試験（predict）」が、たった数行で体験できます。

💡 ヒント：「DataFrame の特定の1行を取り出してモデルで予測する方法」を Copilot に聞いてみましょう。

In [ ]:
# テストデータの1件目を取り出して、モデルの予測と実際の品種を比べる
sample = X_test.iloc[[0]]
pred   = model.predict(sample)
actual = y_test.iloc[0]
print(f"予測: {pred[0]}")
print(f"実際: {actual}")
print(f"結果: {'正解 ✅' if pred[0] == actual else '不正解 ❌'}")

---
### 📊 出力結果の解説

```
予測: wine_0
実際: wine_0
結果: 正解 ✅
```

📌 ポイント：
- `iloc[[0]]` → DataFrame として1行を取り出す（`[0]` だけだと Series になりエラーになる）
- `model.predict()` は「学習済みのルール」を使って予測を返す
- この「fit → predict」の型は Ch.4・Ch.5 でも全く同じ形で使います

✅ （模範解答）ほとんどの場合「正解 ✅」と表示される。不正解の場合は混同行列で確認する。

📌 講師メモ：「たった数行のコードでAIが化学成分から品種を当てた。これがモデルの仕事です」と印象付ける。
「このfit → predict の型を覚えておいてください。Ch.4 でもまったく同じ型を使います」と予告する。

---
## 問2  テストデータで予測して正解率を確認する ★★☆

実務でのイメージ：
「新しい顧客データに対してモデルが正しく品種を当てられるか、正解率で評価する」

---

Q：テストデータで予測して、正解率を計算してください

1. `model.predict(X_test)` で予測する
2. `accuracy_score(y_test, y_pred)` で正解率を計算する

💡 ヒント Copilot への相談例：
「scikit-learn で学習済みモデルでテストデータを予測して、正解率を計算したい」

In [ ]:
# テストデータで予測して、正解率（accuracy）を計算する
y_pred = model.predict(X_test)
acc   = accuracy_score(y_test, y_pred)
print(f"正解率: {acc:.3f}  ({acc*100:.1f}%)")

---
### 📊 出力結果の解説

```
正解率: 0.972  (97.2%)
```

| 確認するポイント | 見るべきこと |
|----------------|------------|
| 正解率 ≈ 97% | 36件中 35件正解（1件だけ間違い） |
| 高い理由 | Ch.1・Ch.2 で見たように、品種間の差が明確なデータだったから |

📌 ポイント 97% は非常に高い正解率。実務では 70〜80% でも「使えるモデル」の場合がある。
正解率だけでなく「どこで間違えたか」も重要 → 次の問3（混同行列）へ。

📌 講師メモ: 「もし正解率が 60% だったらどう思う？」と問い返し、目安の感覚を持たせる。

✅ （模範解答）正解率 97% 程度が期待される。「どの品種を間違えたか」は混同行列で確認する。

---
問2-b（追加）  訓練精度と比べて「過学習していないか」確認しましょう

スライドで「訓練99%・テスト60%」の過学習の例を見ました。
今のモデルは訓練データでも予測させて、テスト精度と比べると過学習の有無が確認できます。

📌 差が小さい（3%以内） → 過学習なし：「本番試験でも勉強の成果が出た」
📌 差が大きい（10%以上）→ 過学習の可能性：「暗記しただけで理解していない」

💡 ヒント：「X_train に predict して accuracy_score で訓練精度を計算する方法」を Copilot に聞いてみましょう。

In [ ]:
# 訓練データでも予測して、訓練精度とテスト精度を比較する
train_pred = model.predict(X_train)
train_acc  = accuracy_score(y_train, train_pred)
print(f"訓練精度: {train_acc:.3f}  ({train_acc*100:.1f}%)")
print(f"テスト精度: {acc:.3f}  ({acc*100:.1f}%)")
print(f"差（訓練 - テスト）: {(train_acc - acc)*100:.1f}%")

---
### 📊 出力結果の解説

```
訓練精度: 1.000  (100.0%)
テスト精度: 0.972  (97.2%)
差（訓練 - テスト）: 2.8%
```

| 状態 | 訓練精度 | テスト精度 | 差 | 判断 |
|------|---------|----------|----|------|
| 今回のモデル | 100% | 97% | 3% | 過学習なし（良好） |
| 過学習の例 | 99% | 60% | 39% | 過学習（実務で使えない） |

📌 ポイント：RandomForest は訓練データを完全に覚えられる（訓練精度=100%）が、
テスト精度が 97% あれば差は小さく、汎化性能が高いと判断できます。
スライドの「訓練99%・テスト60%」と対比すると理解が深まります。

✅ （模範解答）訓練: 100%、テスト: 97%、差: 3%。過学習は起きていない良好な状態。

📌 講師メモ：「差が 3% なのでこのモデルは本番でも使えます」と明確に伝える。
続けて「もし差が 30% だったらどう対処する？」と問い返すと過学習の概念が定着する。

---
### 問2 の気づき（AI 禁止：1 行でOK）

Q：正解率は何 % でしたか？目標の 90% を超えていましたか？

>

✅ （模範解答）97% 程度（36件中 35件正解）。目標の 90% を大きく超えている。wine データは品種間の差が明確なため、RandomForest で高精度が出やすい。

📌 講師メモ：「もし 60% だったらどう思う？」と問い返し、正解率の"感覚"を持たせる。

---
## 問3  「どこで間違えたか」と「何が重要か」を確認する ★★☆（最重要）

実務でのイメージ：
「どの品種（顧客セグメント）を間違えやすいか」を特定して改善する

---

Q3-1：混同行列を表示して、どの品種を間違えたか確認してください

混同行列は「実際の品種」×「予測した品種」の表です。
対角線以外のセルに数値があると「間違い」です。

💡 ヒント Copilot への相談例：
「scikit-learn の confusion_matrix で混同行列を計算して、pandas の DataFrame で見やすく表示したい」

In [ ]:
# 混同行列を計算して、どの品種を間違えたかを確認する
cm     = confusion_matrix(y_test, y_pred)
classes = model.classes_
cm_df   = pd.DataFrame(cm, index=classes, columns=classes)
print("混同行列（行=実際、列=予測）")
print(cm_df)

---
Q3-1（追加）  混同行列を「表形式」で見やすく表示しましょう

`confusion_matrix()` の出力は数値の配列で見づらいです。
品種名をラベルにした DataFrame に変換すると「どの品種をどの品種と間違えたか」が一目でわかります。

💡 ヒント：「confusion_matrix の結果を品種名付きの DataFrame に変換する方法」を Copilot に聞いてみましょう。

In [ ]:
# 混同行列を品種名付きの DataFrame に変換して、どのクラスを間違えたかを見やすく確認する
labels = sorted(y.unique())
cm_df  = pd.DataFrame(confusion_matrix(y_test, y_pred),
                      index=[f"実際:{l}" for l in labels],
                      columns=[f"予測:{l}" for l in labels])
print(cm_df)

---
### 📊 出力結果の解説

```
          予測:wine_0  予測:wine_1  予測:wine_2
実際:wine_0         13           0           0
実際:wine_1          0          20           1
実際:wine_2          0           0          12 （または 1件の誤分類）
```

📌 ポイント：対角線（左上から右下）の数値 = 正解。対角線以外 = 間違い。
「wine_1 を wine_2 と間違えた 1件」が何故か → Ch.2 の散布図で wine_1 と wine_2 が近い場所にある点と一致。

✅ （模範解答）wine_0 は完全正解。wine_1 と wine_2 が 1件ずつ誤分類される場合が多い。

---
### 📊 出力結果の解説

```
混同行列（行=実際、列=予測）
        wine_0  wine_1  wine_2
wine_0      14       0       0
wine_1       0      14       0
wine_2       0       1       7
```

| 読み方 | 内容 |
|--------|------|
| 行（横）| 実際の品種 |
| 列（縦）| モデルが予測した品種 |
| 対角線の数値 | 正解件数（大きいほどよい） |
| 対角線以外 | 間違い件数（ゼロが理想） |

この例では wine_2 を 1 件だけ wine_1 と間違えた。

📌 ポイント どの品種がどの品種に間違われやすいかが見える = 改善の方向性が見える。
「wine_2 の予測精度を上げるには、wine_2 だけのデータを増やす or 特徴量を追加する」と考えられる。

📌 講師メモ: 「wine_1 と wine_2 はどの変数が似ていましたか？（Ch.2 を振り返る）」と発問。

✅ （模範解答）対角線の値が最大になっていれば正しい。wine_2 が 1件 wine_1 と誤分類されることが多い。

---
### Q3-1 の気づき（AI 禁止：1 行でOK）

Q：対角線以外のセルに数値はありましたか？あった場合、どの品種を間違えていましたか？

>

✅ （模範解答）wine_2 が 1件 wine_1 と誤分類された。wine_2 と wine_1 はアルコール度数が近いため。Ch.2 の箱ひげ図で重なりがあると気づいたペアと一致する可能性がある。

📌 講師メモ：「Ch.2 でどの品種が見分けにくいと書きましたか？」と Ch.2 の記録を振り返らせる。

---
Q3-2：特徴量重要度を棒グラフで表示してください

RandomForest は「どの変数が品種の分類に役立ったか（重要度）」を自動的に計算します。
Ch.2 で立てた仮説（alcohol・proline が重要そう）は当たっていましたか？

💡 ヒント Copilot への相談例：
「scikit-learn の RandomForest で特徴量重要度を取得して、棒グラフで表示したい。重要度が高い順に並べたい」

In [ ]:
# 特徴量重要度と特徴量名をセットにした DataFrame を作る
importances = model.feature_importances_
feat_df = pd.DataFrame({"特徴量": X.columns, "重要度": importances})


In [ ]:
# 横棒グラフ用に重要度の小さい順に並べ替える（グラフは下から上に描かれるため大きいものが上になる）
feat_df = feat_df.sort_values("重要度", ascending=True)


In [ ]:
# 特徴量重要度を横棒グラフで表示する
plt.figure(figsize=(8, 6))
plt.barh(feat_df["特徴量"], feat_df["重要度"])
plt.title("特徴量重要度（RandomForest）")
plt.xlabel("重要度")
plt.tight_layout()
plt.show()


---
Q3-2（追加）  重要度の数値を上位5件で確認しましょう

グラフで「どの変数が重要か」の傾向を確認したら、具体的な数値も確認しましょう。
1位と2位の差は大きいですか？Ch.2 の仮説（proline・alcohol が重要）は当たっていましたか？

💡 ヒント：「特徴量重要度を変数名とセットで上位5件を表示する方法」を Copilot に聞いてみましょう。

In [ ]:
# 特徴量重要度の DataFrame を作成して、重要度の大きい順に並べ替える
feat_df = pd.DataFrame({"変数": X.columns, "重要度": model.feature_importances_})
feat_df = feat_df.sort_values("重要度", ascending=False)


In [ ]:
# 上位5件を仮説と照合する
print("特徴量重要度 上位5:")
print(feat_df.head(5).to_string(index=False))


---
### 📊 出力結果の解説

```
      変数    重要度
   proline    0.28
   color_intensity  0.14
   flavanoids  0.13
   alcohol    0.10
   od280/od315  0.08
```

📌 ポイント：proline が圧倒的1位。Ch.2 で「proline の箱ひげ図が品種間で全く重なっていなかった」という視覚的な仮説が、モデルの数値でも確認されました。
→ 「EDA → 仮説 → モデルで検証」というデータ分析の基本サイクルが完結する瞬間です。

✅ （模範解答）proline が1位（重要度 0.25〜0.35 程度）。Ch.2 で立てた仮説と一致。alcohol は4〜5位程度。

📌 講師メモ：「Ch.2 で箱ひげ図を見たとき、wine_0 の proline が他と全く重ならなかった。あれが今の 1 位に繋がっています」と言葉で繋げてあげると理解が深まります。

---
### 📊 出力結果の解説

上位 3 変数（目安）：
1. proline（最重要）
2. color_intensity
3. flavanoids

| Ch.2 での仮説 | 結果 | 一致？ |
|-------------|------|--------|
| proline が重要そう | 1位 | 一致 |
| alcohol が重要そう | 4〜5位 | おおむね一致 |
| flavanoids が重要そう | 3位 | 一致 |

📌 ポイント グラフで立てた仮説が機械学習で「数値として証明された」 = EDA の価値。
重要度上位の変数を「なぜこの変数が重要か」とビジネス視点で解釈することが実務で大切。

📌 講師メモ: 「Ch.2 の仮説は当たっていましたか？」と問い返す。

✅ （模範解答）proline が最重要。Ch.2 で「ばらつきが最大で品種間の差が最大」と気づいた変数と一致する。
EDA（探索的データ分析）が機械学習の結果と一致することで、分析の整合性が確認できる。

---
### Q3-2 の気づき（AI 禁止：1 行でOK）

Q：特徴量重要度 1 位の変数は何でしたか？Ch.2 の仮説は当たっていましたか？

>

✅ （模範解答）proline が 1位。Ch.2 で「std（標準偏差）が最大で品種間の差が最大」と気づいた変数と一致する。Ch.1 → Ch.2 → Ch.3 の分析の流れが整合している。

📌 講師メモ：「グラフで立てた仮説が数値で証明された」ことを強調し、EDA の価値を印象付ける。

---
問3 まとめ（追加）  全評価指標を一度に確認しましょう ★★☆

`classification_report` を使うと Precision・Recall・F1スコアを品種ごとに一覧で確認できます。
問4（発展）の前に全体像を俯瞰しておきましょう。

💡 ヒント：「classification_report で分類の詳細指標を一覧表示する方法」を Copilot に聞いてみましょう。

In [ ]:
# Precision・Recall・F1スコアを一覧で確認する
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))


---
## STEP 3  考察・振り返り （AI 禁止）

⛔ 注意 AI は使わないこと。自分の言葉で書いてください。

Q：Ch.2 で立てた仮説（重要な変数の予想）は当たっていましたか？

>

Q：正解率が約 97% というのは「良いモデル」だと思いますか？その理由は？

💡 参考：実務では 70〜80% でも「使えるモデル」として採用されることがあります。何 % 以上が合格かはビジネスの文脈によります。

>

---
### 📌 講師メモ（考察の模範解答）

（模範解答 1）仮説の検証：
proline・flavanoids は Ch.2 の仮説通り。
alcohol は予想より低い順位だった場合 → 「alcohol だけでは品種を分けにくかったが、他の変数と組み合わせると有効」と解説。

（模範解答 2）97% の評価：
良いモデルと言える。ただし今回はデータが 178件と少なく、品種間の差も明確だったため高く出やすい。
実務では 70〜80% でも十分に価値があるケースも多い。「何%以上が合格か」はビジネスの文脈によって異なる。

よくある受講生の詰まり方：
- 「97% は低い？」→ 「何件中何件正解か計算してみよう（36件中 35件正解）」と具体化する

In [ ]:
# 考察を記入したら、問4（発展）へ進んでください（時間が余った人のみ）

---
### 📊 出力結果の解説

```
              precision    recall  f1-score   support

      wine_0       1.00      1.00      1.00        14
      wine_1       0.95      1.00      0.97        20
      wine_2       1.00      0.88      0.93         8

    accuracy                           0.97        42
   macro avg       0.98      0.96      0.97        42
weighted avg       0.98      0.97      0.97        42
```

| 列名 | 意味 |
|------|------|
| precision | 「その品種と予測した中で本当にその品種だった割合」 |
| recall | 「本当にその品種のうち正しく予測できた割合」 |
| f1-score | precision と recall の調和平均（バランス指標） |
| support | テストデータ内のその品種の実際の件数 |

📌 ポイント：`classification_report` は1行で全指標を確認できる便利なコマンド。
wine_2 の recall が 0.88 と最も低い（1件見逃し）ことが読み取れます。
問4の P/R 計算は「この表の数値を自分で計算する練習」です。

✅ （模範解答）wine_2 の recall が最低（1件見逃し）。wine_1 の precision が 0.95 = 予測した wine_1 のうち 1件は別品種。

📌 講師メモ：問4への橋渡しとして「今の表の precision と recall を自分で計算してみましょう」と接続する。

---
## 問4（発展）  予測の詳細を確認する ★★★

📋 発展 時間が余った人だけ取り組んでください

実務でのイメージ：

📌 **Precision（適合率）**：「wine_X と予測した中で、本当に wine_X だった割合」
　例）「陽性と診断した患者のうち、本当に病気だった割合」

📌 **Recall（再現率）**：「本当に wine_X のうち、正しく wine_X と予測できた割合」
　例）「本当に病気の患者のうち、正しく陽性と診断できた割合」

→ 「見逃したくない（病気なのに陰性と言いたくない）」→ **Recall を高める**
→ 「誤診をなくしたい（病気でないのに陽性と言いたくない）」→ **Precision を高める**

---

Q：品種ごとの Precision・Recall を計算して表示してください

💡 ヒント Copilot への相談例：
「scikit-learn で分類モデルの品種ごとの Precision と Recall を計算したい（average='macro'）」

In [ ]:
# 品種ごとの Precision・Recall を計算して確認する
from sklearn.metrics import precision_score, recall_score
prec   = precision_score(y_test, y_pred, average=None)
rec    = recall_score(y_test, y_pred, average=None)
pr_df  = pd.DataFrame({"品種": model.classes_, "Precision": prec.round(3), "Recall": rec.round(3)})
print(pr_df)

---
### 📊 出力結果の解説

```
     品種  Precision  Recall
0  wine_0      1.000   1.000
1  wine_1      0.933   1.000
2  wine_2      1.000   0.875
```

| 指標 | 意味 | この結果 |
|------|------|---------|
| Precision | 「wine_X と予測した中で、本当に wine_X だった割合」 | wine_1 が 0.93 → 予測の 7% は誤り |
| Recall | 「本当に wine_X のうち、正しく wine_X と予測できた割合」 | wine_2 が 0.875 → 1件見逃し |

✅ （模範解答）wine_2 の Recall が 0.875 = 実際の wine_2 のうち 12.5%（1件）を見逃している。
wine_1 の Precision が 0.933 = wine_1 と予測した中の 6.7%（1件）は実際は別品種だった。

---
## お疲れさまでした！

| 操作 | 発見 | ビジネスへの接続 |
|------|------|----------------|
| モデル学習 | 13 変数で自動的にルールを作成 | 人間のルール作りを自動化 |
| 正解率 97% | ほぼすべての品種を正しく分類 | 現場で使えるモデルの基準 |
| 混同行列 | wine_2 の一部が wine_1 に誤分類 | 改善が必要な品種がわかる |
| 特徴量重要度 | proline が最重要 → Ch.2 の仮説と一致 | EDA → ML の流れが完結 |